### Install requiremnts

In [1]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from checkpoints import DT, MT, LT
import os
from validate import load_model, get_test_loader, test
from checkpoints import DT, MT, LT
import numpy as np
import subprocess


DOWN_TASKS = ["HSN", "POW", "SNE", "PER", "NES", "UHH", "NBP", "SSW"]

/home/bellafkir/Documents/sa4birds/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Download checkpoints

In [3]:

all_checkpoint_available = (
    all(os.path.exists(p) for ds in DT for p in DT[ds]) and
    all(os.path.exists(p) for p in MT['ALL']) and
    all(os.path.exists(p) for p in LT['ALL'])
)

if not all_checkpoint_available:
    subprocess.run(["python", "prepare_checkpoints.py"])
else:
    print("All checkpoints available")

All checkpoints available


In [4]:
def run_testing(regime, device='cuda'):
    for down_task in DOWN_TASKS:

        # Get checkpoint paths for this task (empty list if task not present)
        ckpts = regime.get(down_task, [])
        if not ckpts:
            ckpts  = regime['ALL']

        print(ckpts)

        models = []
        configs = []

        # ---------------------------------------------------------
        # Load models and their configs from checkpoints
        # ---------------------------------------------------------
        for ckpt in ckpts:
            m, cfg = load_model(ckpt, device=device)
            models.append(m)
            # Override validation parameters for evaluation
            cfg.frontend.val_target_length = 701
            cfg.event_decoder.val.extracted_interval = 7
            # Set dataset name to current downstream task
            cfg.train.dataset_name = down_task
            configs.append(cfg)

        results = dict()
        # Create test dataloader using the first config
        val_loader, class_names = get_test_loader(configs[0])

        # Map class names to label IDs using the config label map
        label2id = {k: v for k, v in configs[0].train.label_map.items()}

        # Only test on classes relevant for the current dataset
        relevant = [label2id[c] for c in class_names]

        # Store metrics for each model (for later averaging)
        metrics = {"auroc": [],
                   "cmap": [],
                   "top1_acc": []}

        # ---------------------------------------------------------
        # test each model on the test dataset
        # ---------------------------------------------------------
        for model, _ in zip(models, configs):
            # Run evaluation
            auroc, cmap, top1_acc = test(model, val_loader, relevant, device=device)

            # Store metrics
            metrics["auroc"].append(auroc)
            metrics["cmap"].append(cmap)
            metrics["top1_acc"].append(top1_acc)

        # ---------------------------------------------------------
        # Aggregate metrics across all checkpoints/models
        # ---------------------------------------------------------
        results[down_task] = {key: float(np.mean(values)) for key, values in metrics.items()}
        print(results)

In [6]:
# DT REGIME
# for cpu: run_testing(DT, device='cpu')
run_testing(DT)

['checkpoints/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_112131', 'checkpoints/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_113316', 'checkpoints/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_114501']


2026-03-06 14:48:14,900 | INFO | Task: HSN | Number of events: 12000
Testing: 100%|██████████| 94/94 [00:46<00:00,  2.04it/s]


{'HSN': {'auroc': 0.9582019252201605, 'cmap': 0.5805238184761927, 'top1_acc': 0.7330605586369833}}
['checkpoints/DT/POW/POW_eca_nfnet_l1_2025-10-11_143357', 'checkpoints/DT/POW/POW_eca_nfnet_l1_2025-10-11_145846', 'checkpoints/DT/POW/POW_eca_nfnet_l1_2025-10-11_152335']


2026-03-06 14:50:34,650 | INFO | Task: POW | Number of events: 4560
Testing: 100%|██████████| 36/36 [00:24<00:00,  1.46it/s]


{'POW': {'auroc': 0.9422970928184865, 'cmap': 0.6007718514368877, 'top1_acc': 0.9446458220481873}}
['checkpoints/DT/SNE/SNE_eca_nfnet_l1_2025-10-12_131606', 'checkpoints/DT/SNE/SNE_eca_nfnet_l1_2025-10-12_135017', 'checkpoints/DT/SNE/SNE_eca_nfnet_l1_2025-10-12_142429']


...

{'UHH': {'auroc': 0.883188941937096, 'cmap': 0.344212708470556, 'top1_acc': 0.6776004433631897}}
['checkpoints/DT/NBP/NBP_eca_nfnet_l1_2025-10-13_143637', 'checkpoints/DT/NBP/NBP_eca_nfnet_l1_2025-10-13_151856', 'checkpoints/DT/NBP/NBP_eca_nfnet_l1_2025-10-13_160131']


2026-03-06 15:09:47,677 | INFO | Task: NBP | Number of events: 539
Testing: 100%|██████████| 5/5 [00:05<00:00,  1.03s/it]


{'NBP': {'auroc': 0.9274592879180369, 'cmap': 0.7067440309580988, 'top1_acc': 0.7056277195612589}}
['checkpoints/DT/SSW/SSW_eca_nfnet_l1_2025-10-11_215626', 'checkpoints/DT/SSW/SSW_eca_nfnet_l1_2025-10-11_203307', 'checkpoints/DT/SSW/SSW_eca_nfnet_l1_2025-10-11_190939']


2026-03-06 15:10:05,482 | INFO | Task: SSW | Number of events: 205200
Testing: 100%|██████████| 1604/1604 [10:54<00:00,  2.45it/s]


{'SSW': {'auroc': 0.9761287101627084, 'cmap': 0.5188967385558886, 'top1_acc': 0.7864448229471842}}


In [7]:
run_testing(MT)

['checkpoints/MT/MT_eca_nfnet_l1_2025-11-25_151907', 'checkpoints/MT/MT_eca_nfnet_l1_2025-11-25_214315', 'checkpoints/MT/MT_eca_nfnet_l1_2025-11-26_010815']


2026-03-06 15:45:08,390 | INFO | Task: HSN | Number of events: 12000
Testing: 100%|██████████| 94/94 [00:46<00:00,  2.01it/s]


{'HSN': {'auroc': 0.9184371456266923, 'cmap': 0.5579475516695079, 'top1_acc': 0.7075196901957194}}
['checkpoints/MT/MT_eca_nfnet_l1_2025-11-25_151907', 'checkpoints/MT/MT_eca_nfnet_l1_2025-11-25_214315', 'checkpoints/MT/MT_eca_nfnet_l1_2025-11-26_010815']


2026-03-06 15:47:30,905 | INFO | Task: POW | Number of events: 4560
Testing: 100%|██████████| 36/36 [00:23<00:00,  1.52it/s]


{'POW': {'auroc': 0.9375645350612535, 'cmap': 0.5368986922681116, 'top1_acc': 0.9144389033317566}}
['checkpoints/MT/MT_eca_nfnet_l1_2025-11-25_151907', 'checkpoints/MT/MT_eca_nfnet_l1_2025-11-25_214315', 'checkpoints/MT/MT_eca_nfnet_l1_2025-11-26_010815']


...

{'UHH': {'auroc': 0.885294418690198, 'cmap': 0.3412588201287024, 'top1_acc': 0.7122304836908976}}
['checkpoints/MT/MT_eca_nfnet_l1_2025-11-25_151907', 'checkpoints/MT/MT_eca_nfnet_l1_2025-11-25_214315', 'checkpoints/MT/MT_eca_nfnet_l1_2025-11-26_010815']


2026-03-06 16:06:54,706 | INFO | Task: NBP | Number of events: 539
Testing: 100%|██████████| 5/5 [00:05<00:00,  1.03s/it]


{'NBP': {'auroc': 0.935165767197601, 'cmap': 0.6946172203238211, 'top1_acc': 0.7167594234148661}}
['checkpoints/MT/MT_eca_nfnet_l1_2025-11-25_151907', 'checkpoints/MT/MT_eca_nfnet_l1_2025-11-25_214315', 'checkpoints/MT/MT_eca_nfnet_l1_2025-11-26_010815']


2026-03-06 16:07:12,409 | INFO | Task: SSW | Number of events: 205200
Testing: 100%|██████████| 1604/1604 [11:03<00:00,  2.42it/s]


{'SSW': {'auroc': 0.9728458534436427, 'cmap': 0.48455712328995015, 'top1_acc': 0.7581257025400797}}


In [8]:
run_testing(LT)

['checkpoints/LT/LT_eca_nfnet_l1_2025-11-25_043730', 'checkpoints/LT/LT_eca_nfnet_l1_2025-11-25_060339', 'checkpoints/LT/LT_eca_nfnet_l1_2025-11-24_180849']


2026-03-06 16:41:08,795 | INFO | Task: HSN | Number of events: 12000
Testing: 100%|██████████| 94/94 [00:56<00:00,  1.67it/s]


{'HSN': {'auroc': 0.9362847044702539, 'cmap': 0.5451335159817338, 'top1_acc': 0.7223978638648987}}
['checkpoints/LT/LT_eca_nfnet_l1_2025-11-25_043730', 'checkpoints/LT/LT_eca_nfnet_l1_2025-11-25_060339', 'checkpoints/LT/LT_eca_nfnet_l1_2025-11-24_180849']


2026-03-06 16:44:02,799 | INFO | Task: POW | Number of events: 4560
Testing: 100%|██████████| 36/36 [00:27<00:00,  1.33it/s]


{'POW': {'auroc': 0.9054063059801476, 'cmap': 0.5424785885572563, 'top1_acc': 0.9435130556424459}}
['checkpoints/LT/LT_eca_nfnet_l1_2025-11-25_043730', 'checkpoints/LT/LT_eca_nfnet_l1_2025-11-25_060339', 'checkpoints/LT/LT_eca_nfnet_l1_2025-11-24_180849']


...

{'UHH': {'auroc': 0.8974533207180894, 'cmap': 0.3206880047688036, 'top1_acc': 0.6917312343915304}}
['checkpoints/LT/LT_eca_nfnet_l1_2025-11-25_043730', 'checkpoints/LT/LT_eca_nfnet_l1_2025-11-25_060339', 'checkpoints/LT/LT_eca_nfnet_l1_2025-11-24_180849']


2026-03-06 17:08:16,589 | INFO | Task: NBP | Number of events: 539
Testing: 100%|██████████| 5/5 [00:05<00:00,  1.13s/it]


{'NBP': {'auroc': 0.9368651411143865, 'cmap': 0.7066477922374909, 'top1_acc': 0.7322201530138651}}
['checkpoints/LT/LT_eca_nfnet_l1_2025-11-25_043730', 'checkpoints/LT/LT_eca_nfnet_l1_2025-11-25_060339', 'checkpoints/LT/LT_eca_nfnet_l1_2025-11-24_180849']


2026-03-06 17:08:38,635 | INFO | Task: SSW | Number of events: 205200
Testing: 100%|██████████| 1604/1604 [14:07<00:00,  1.89it/s]


{'SSW': {'auroc': 0.9696315710574622, 'cmap': 0.45494037936116544, 'top1_acc': 0.7615264256795248}}
